# ISRO BAH 2026 — PS-07: Notebook 4: Full Preprocessing (All 60k Stars)

**Purpose**: Apply the complete preprocessing pipeline to all 60k TESS light curves across Sectors 1–3.  
**Requirements**: PREP-01..07  
**Input datasets**: `tess-sector1-raw`, `tess-sector2-raw`, `tess-sector3-raw` (mount in Kaggle settings)  
**Estimated runtime**: 3–4 hours (4-core Kaggle T4 instance, parallel)  
**Output**: Kaggle Dataset `tess-preprocessed`

**Pipeline per star**:
1. PREP-01: Quality mask (bitmask 175)
2. PREP-06: Exclusion filter (Tmag<6, <500 cadences)
3. PREP-02: 5σ sigma clip + normalise to median=1.0
4. PREP-05: Gap mask (13-day gaps flagged)
5. PREP-03: Biweight detrend (Wotan, 0.75d window)
6. PREP-07: Limb darkening extraction (TICv8 + Claret 2017)
7. Save preprocessed .npz

In [ ]:
%pip install -q "lightkurve>=2.4" "astroquery>=0.4.7" "pyarrow>=14" "wotan>=1.10" "celerite2>=0.3"


In [ ]:
import sys, os
from pathlib import Path

PIPELINE_PATH = Path('/kaggle/input/isro-bah-pipeline')
if PIPELINE_PATH.exists():
    sys.path.insert(0, str(PIPELINE_PATH))
else:
    CLONE_DIR = Path('/kaggle/working/ISRO-BAH')
    if not CLONE_DIR.exists():
        GITHUB_URL = 'https://github.com/YOUR_USERNAME/ISRO-BAH.git'  # ← UPDATE THIS
        subprocess.run(['git', 'clone', '--depth=1', GITHUB_URL, str(CLONE_DIR)], check=True)
    sys.path.insert(0, str(CLONE_DIR))

import pipeline.config as cfg

# Input: raw .npz files from sector datasets
# Raw data is in /kaggle/input/tess-sector1-raw/raw/, etc.
# We'll copy them to working dir (or load directly from input)

KAGGLE_OUT = Path('/kaggle/working')
cfg.OUTPUTS_DIR   = KAGGLE_OUT / 'outputs'
cfg.RAW_DIR       = cfg.OUTPUTS_DIR / 'raw'         # will be populated from input datasets
cfg.PREP_DIR      = cfg.OUTPUTS_DIR / 'preprocessed'
cfg.CATALOGUE_DIR = cfg.OUTPUTS_DIR / 'catalogue'
cfg.LD_TABLE_PATH = cfg.CATALOGUE_DIR / 'claret_ld.parquet'
cfg.MASTER_CATALOGUE_PATH = cfg.CATALOGUE_DIR / 'master_catalogue.parquet'

for d in [cfg.RAW_DIR, cfg.PREP_DIR, cfg.CATALOGUE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Config ready. PREP_DIR:', cfg.PREP_DIR)

In [ ]:
# ── Symlink/copy raw .npz from sector input datasets ──────────────────────
import shutil

for sector in [1, 2, 3]:
    input_dir = Path(f'/kaggle/input/tess-sector{sector}-raw/outputs/raw')
    if input_dir.exists():
        files = list(input_dir.glob(f'*_s{sector}.npz'))
        print(f'Sector {sector}: {len(files)} raw .npz files found in input dataset')
        # Create symlinks to avoid copying 6 GB × 3 sectors
        for f in files:
            link = cfg.RAW_DIR / f.name
            if not link.exists():
                link.symlink_to(f)
    else:
        print(f'⚠ Sector {sector} input dataset not found at {input_dir}')
        print(f'  → Add "tess-sector{sector}-raw" as input dataset in Kaggle settings')

total = len(list(cfg.RAW_DIR.glob('*.npz')))
print(f'\nTotal raw .npz files accessible: {total}')

In [ ]:
# ── Load master catalogue (merged from sector notebooks) ──────────────────
import pandas as pd
import pyarrow.parquet as pq

cat_dfs = []
for sector in [1, 2, 3]:
    cat_path = Path(f'/kaggle/input/tess-sector{sector}-raw/outputs/catalogue/master_catalogue.parquet')
    if cat_path.exists():
        cat_dfs.append(pd.read_parquet(str(cat_path)))
        print(f'Sector {sector} catalogue: {len(cat_dfs[-1])} rows')

if cat_dfs:
    cat = pd.concat(cat_dfs, ignore_index=True)
    cat = cat.drop_duplicates(subset=['tic_id', 'sector'], keep='last')
    pq.write_table(
        pq.read_table(str(cat_dfs[0].to_parquet('/tmp/cat.parquet', index=False) or '/tmp/cat.parquet')),
        str(cfg.MASTER_CATALOGUE_PATH)
    )
    # Simpler:
    cat.to_parquet(str(cfg.MASTER_CATALOGUE_PATH), index=False)
    print(f'\nMerged catalogue: {len(cat)} stars across 3 sectors')
else:
    print('⚠ No catalogues found. Check input datasets.')

In [ ]:
# ── Copy Claret LD table from pipeline dataset ─────────────────────────────
# The LD table is committed to the repo (small ~1 MB)
ld_source = PIPELINE_PATH / 'outputs' / 'catalogue' / 'claret_ld.parquet'
if ld_source.exists():
    shutil.copy(str(ld_source), str(cfg.LD_TABLE_PATH))
    print(f'Claret LD table copied from pipeline dataset: {cfg.LD_TABLE_PATH}')
else:
    print('⚠ Claret LD table not found — limb darkening will use defaults.')
    print('  Run scripts/download_ld_table.py locally and commit the output.')

In [ ]:
# ── Preprocessing worker function (parallel-safe) ─────────────────────────
import numpy as np
from pipeline.ingest.store              import load_lc_npz, save_lc_npz, npz_path
from pipeline.preprocess.quality_mask  import apply_quality_mask
from pipeline.preprocess.sigma_clip    import apply_sigma_clip
from pipeline.preprocess.gap_mask      import apply_gap_mask
from pipeline.preprocess.detrend       import apply_biweight_detrend
from pipeline.preprocess.filters       import should_process, get_exclusion_reason
from pipeline.preprocess.limb_darkening import get_ld_coefficients

def preprocess_star(args):
    """Process a single star — designed for multiprocessing.Pool."""
    tic_id, sector, Tmag = args
    
    # Check if already preprocessed
    out_path = npz_path(tic_id, sector, kind='preprocessed')
    if out_path.exists():
        return ('SKIPPED', tic_id, sector, 'NONE')
    
    try:
        lc = load_lc_npz(tic_id, sector, kind='raw')
    except FileNotFoundError:
        return ('MISSING', tic_id, sector, 'NO_RAW_FILE')
    except Exception as e:
        return ('ERROR', tic_id, sector, str(e))
    
    try:
        # PREP-01: Quality mask
        time_arr, flux_arr, flux_err, quality_arr = apply_quality_mask(
            lc['time'], lc['flux'], lc['flux_err'], lc['quality']
        )
        
        # PREP-06: Exclusion filter
        reason = get_exclusion_reason(Tmag, len(time_arr))
        if reason != 'NONE':
            return ('EXCLUDED', tic_id, sector, reason)
        
        # PREP-02: Sigma clip + normalise
        time_arr, flux_arr, flux_err, _ = apply_sigma_clip(time_arr, flux_arr, flux_err)
        
        # PREP-05: Gap mask
        gap_mask, gaps = apply_gap_mask(time_arr)
        
        # PREP-03: Biweight detrend
        flux_detrended, trend = apply_biweight_detrend(time_arr, flux_arr)
        
        # PREP-07: Limb darkening
        meta = lc['meta'].copy()
        u1, u2 = get_ld_coefficients(tic_id, meta)
        meta['u1'] = u1
        meta['u2'] = u2
        meta['n_cadences_clean'] = len(time_arr)
        meta['gaps_btjd'] = [(float(g[0]), float(g[1])) for g in gaps]
        
        # Save
        q_save = quality_arr[:len(time_arr)] if len(quality_arr) >= len(time_arr) else np.zeros(len(time_arr), dtype=np.int32)
        out_path = save_lc_npz(tic_id, sector, time_arr, flux_detrended, flux_err, q_save, meta, kind='preprocessed')
        
        # Write gap_mask into the npz
        data = dict(np.load(str(out_path), allow_pickle=True))
        data['gap_mask'] = gap_mask
        np.savez_compressed(str(out_path), **data)
        
        return ('SUCCESS', tic_id, sector, reason)
    
    except Exception as e:
        return ('ERROR', tic_id, sector, str(e)[:200])

print('Worker function defined.')

In [ ]:
# ── MAIN: Parallel preprocessing ─────────────────────────────────────────
import multiprocessing as mp
import time
from collections import Counter

# Build star list from catalogue
star_list = [
    (int(row['tic_id']), int(row['sector']), float(row.get('Tmag', float('nan'))))
    for _, row in cat.iterrows()
    if row['download_status'] == 'SUCCESS'
]
print(f'Stars to process: {len(star_list)}')
print(f'Using {mp.cpu_count()} CPU cores')

WORKERS = mp.cpu_count()  # 4 on Kaggle T4
counts = Counter()
start = time.time()

with mp.Pool(processes=WORKERS) as pool:
    for i, (status, tic_id, sector, reason) in enumerate(
        pool.imap_unordered(preprocess_star, star_list), start=1
    ):
        counts[status] += 1
        if i % 1000 == 0 or i == len(star_list):
            elapsed = time.time() - start
            rate = i / elapsed
            eta = (len(star_list) - i) / rate if rate > 0 else 0
            print(f'{i}/{len(star_list)} ({100*i/len(star_list):.1f}%) | '
                  f'{rate*60:.0f}/min | ETA {eta/60:.0f}min | {dict(counts)}')

elapsed = time.time() - start
print(f'\n✓ Preprocessing complete in {elapsed/60:.1f} min')
print(dict(counts))

In [ ]:
# ── Validate: WASP-121b transit preservation ──────────────────────────────
# Critical: verify detrending does NOT erase WASP-121b's 1.4% transit depth
from pipeline.preprocess.detrend import validate_transit_preservation
import matplotlib.pyplot as plt

WASP121_TIC = 22529346
try:
    lc_prep = load_lc_npz(WASP121_TIC, sector=1, kind='preprocessed')
    
    # WASP-121b parameters (Delrez+2016)
    P_wasp121   = 1.27492    # days
    T0_wasp121  = 1325.5     # BTJD approx (adjust to actual Sector 1 T0)
    dur_wasp121 = 0.12       # days (~2.9h)
    depth_wasp121 = 0.014    # 1.4%
    
    passed = validate_transit_preservation(
        lc_prep['time'], lc_prep['flux'],
        period=P_wasp121, t0=T0_wasp121,
        duration_days=dur_wasp121,
        expected_depth=depth_wasp121,
        label='WASP-121b',
    )
    
    # Phase-fold plot
    phase = ((lc_prep['time'] - T0_wasp121) % P_wasp121) / P_wasp121
    phase[phase > 0.5] -= 1.0
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(lc_prep['time'], lc_prep['flux'], 'k.', ms=0.5, alpha=0.3)
    axes[0].set_xlabel('BTJD'); axes[0].set_ylabel('Detrended Flux')
    axes[0].set_title('WASP-121b — Detrended LC')
    
    axes[1].plot(phase, lc_prep['flux'], 'k.', ms=1, alpha=0.3)
    axes[1].set_xlim(-0.15, 0.15)
    axes[1].set_xlabel('Phase'); axes[1].set_ylabel('Detrended Flux')
    axes[1].set_title(f'WASP-121b — Phase folded (P={P_wasp121}d) — Depth preserved: {passed}')
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/wasp121b_detrended.png', dpi=150)
    plt.show()
    
    if passed:
        print('✓ WASP-121b transit depth preserved after biweight detrending.')
    else:
        print('✗ WASP-121b depth NOT preserved — check window_length parameter.')
        print('  Try increasing BIWEIGHT_WINDOW or using a transit mask.')
        
except FileNotFoundError:
    print('✗ WASP-121b preprocessed file not found.')

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────
n_prep = len(list(cfg.PREP_DIR.glob('*.npz')))
size_gb = sum(f.stat().st_size for f in cfg.PREP_DIR.glob('*.npz')) / 1e9
print(f'Preprocessed: {n_prep} light curves | {size_gb:.2f} GB')
print('\nNext: Upload outputs/ as "tess-preprocessed" dataset, then run nb05_tls_search.ipynb')